In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [3]:
def print_messages(messages: list[any]) -> None:
    for message in messages:
        message.pretty_print()

In [4]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [5]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [6]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password."""

    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(
            update={
                "authenticated": True,
                "messages": [
                    ToolMessage("Successfully authenticated", tool_call_id=runtime.tool_call_id)
                ]
            }
        )
    else:
        return Command(
            update={
                "authenticated": False,
                "messages": [
                    ToolMessage("Authentication faileid", tool_call_id=runtime.tool_call_id)
                ]
            }
        )

@tool
def check_inbox(runtime: ToolRuntime) -> str:
    """Check the inbox for recent emails"""

    return """
    Hi Julie,
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""

    return f"Email sent to {to} with subject {subject} and body {body}"

In [7]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Allow read inbox and send email tools only if user provides correct emain and password"""

    authenticated = request.state.get("authenticated")

    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools)
    return handler(request)

In [8]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = "You are a helpful assistant that can check the inbox and send emails."
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [9]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

email_agent = create_agent(
    model="gpt-5-nano",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call,
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True
            }
        )
    ]
)

In [10]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = email_agent.invoke(
    {"messages": HumanMessage(content="Please check my inbox")},
    context=EmailContext(),
    config=config
)

print_messages(response["messages"])

================================ Human Message =================================

Please check my inbox
================================== Ai Message ==================================

I can help with that. I need to sign you in first. Please provide:

- Email address
- Password

After you authenticate, I can check your inbox and show unread messages, subjects, senders, etc. If you’d prefer not to share credentials here, I can guide you through signing in on your device instead.


In [12]:
response = email_agent.invoke(
    {"messages": HumanMessage(content="julie@example.com, password123")},
    context=EmailContext(),
    config=config
)

print_messages(response["messages"])

/workspaces/lca-lc-foundations/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(email_addres... password='password123'), input_type=EmailContext])
  return self.__pydantic_serializer__.to_python(
/workspaces/lca-lc-foundations/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=EmailContext(email_addres... password='password123'), input_type=EmailContext])
  return self.__pydantic_serializer__.to_python(


================================ Human Message =================================

Please check my inbox
================================== Ai Message ==================================

I can help with that. I need to sign you in first. Please provide:

- Email address
- Password

After you authenticate, I can check your inbox and show unread messages, subjects, senders, etc. If you’d prefer not to share credentials here, I can guide you through signing in on your device instead.
================================ Human Message =================================

julie@example.com, password123
================================== Ai Message ==================================
Tool Calls:
  authenticate (call_fLbbcjSQoO3ADZbRY8DeztkD)
 Call ID: call_fLbbcjSQoO3ADZbRY8DeztkD
  Args:
    email: julie@example.com
    password: password123
================================= Tool Message =================================
Name: authenticate

Successfully authenticated
===============================

In [13]:
response = email_agent.invoke(
    {"messages": HumanMessage(content="reply with casual tone, for next wednesday at 10am")},
    context=EmailContext(),
    config=config
)

print_messages(response["messages"])

================================ Human Message =================================

Please check my inbox
================================== Ai Message ==================================

I can help with that. I need to sign you in first. Please provide:

- Email address
- Password

After you authenticate, I can check your inbox and show unread messages, subjects, senders, etc. If you’d prefer not to share credentials here, I can guide you through signing in on your device instead.
================================ Human Message =================================

julie@example.com, password123
================================== Ai Message ==================================
Tool Calls:
  authenticate (call_fLbbcjSQoO3ADZbRY8DeztkD)
 Call ID: call_fLbbcjSQoO3ADZbRY8DeztkD
  Args:
    email: julie@example.com
    password: password123
================================= Tool Message =================================
Name: authenticate

Successfully authenticated
===============================

In [14]:
print(response["__interrupt__"][0].value["action_requests"][0]["args"]["body"])

Hey Jane!

Coffee sounds great. Wednesday, January 7 at 10:00 AM works for me. Where should we meet? If that works for you, we can stick to our usual spot, or you can pick a place.

See you soon!
Julie


In [25]:
response = email_agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}
    ),
    config=config
)

print_messages(response["messages"])

================================ Human Message =================================

Please check my inbox
================================== Ai Message ==================================

I can help with that. I need to sign you in first. Please provide:

- Email address
- Password

After you authenticate, I can check your inbox and show unread messages, subjects, senders, etc. If you’d prefer not to share credentials here, I can guide you through signing in on your device instead.
================================ Human Message =================================

julie@example.com, password123
================================== Ai Message ==================================
Tool Calls:
  authenticate (call_fLbbcjSQoO3ADZbRY8DeztkD)
 Call ID: call_fLbbcjSQoO3ADZbRY8DeztkD
  Args:
    email: julie@example.com
    password: password123
================================= Tool Message =================================
Name: authenticate

Successfully authenticated
===============================

In [26]:
print(response["messages"][-1].content)

Email sent successfully.

Summary:
- To: jane@example.com
- Subject: Re: Coffee next week
- Body (casual):
  Hey Jane!

  Coffee sounds great. Wednesday, January 7 at 10:00 AM works for me. Where should we meet? If that works for you, we can stick to our usual spot, or you can pick a place.

  See you soon!
  Julie

Would you like me to:
- Add a calendar event or reminder for Wednesday at 10:00 AM?
- Send a quick follow-up if you don’t hear back by a certain date? If so, when should I follow up?


In [27]:
from pprint import pprint

pprint(response)

{'authenticated': True,
 'messages': [HumanMessage(content='Please check my inbox', additional_kwargs={}, response_metadata={}, id='029cffae-e1b0-4807-94a4-5c06060e1bf9'),
              AIMessage(content='I can help with that. I need to sign you in first. Please provide:\n\n- Email address\n- Password\n\nAfter you authenticate, I can check your inbox and show unread messages, subjects, senders, etc. If you’d prefer not to share credentials here, I can guide you through signing in on your device instead.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 844, 'prompt_tokens': 148, 'total_tokens': 992, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Cv6IsiZ1Mb5P2TR2ZXyb30gVielMP